# Multi-agent prospect theory experiment

This course demo uses a language model to simulate a discussion among three advisors and then elicits a decision from each advisor. LangGraph manages the speaking order and conversation history.

- **Scenario** — 600 lives are at risk. The advisors choose between a certain and an uncertain loss.
- **Advisors** — a foreign ministry department director, a think tank analyst, and a navy captain.
- **Procedure** — the advisors take turns speaking for 10 rounds. The notebook then asks each advisor to make an independent choice.

This exercise runs the loss frame. In class, consider how to write the equivalent gain frame and compare outcomes across the two conditions to test for a framing effect.

## Run the notebook

Open the notebook with Python 3.10 or later. Install the dependencies listed in the repository README, then run the cells from top to bottom. Choose a model that supports structured output. A complete run makes 33 model calls.

In [ ]:
# Imports
from typing import Dict, List, Optional, Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

print("Dependencies imported.")

In [ ]:
class CoreDiscussionState(TypedDict):
    """Track the discussion and let LangGraph append messages."""
    messages: Annotated[List[AnyMessage], add_messages]
    current_speaker: Optional[str]
    turn_order: List[str]
    current_turn: int  # Zero-based count of completed turns before routing.

    agents: Dict[str, str]  # Advisor ID to profile.
    scenario: str

    discussion_complete: bool
    max_rounds: int

In [ ]:
def create_core_coordinator(debug: bool = False):
    """Manage speaking order and decide when to end the discussion.

    Consider how the stopping rule could check for consensus or let the agents
    request an end to the discussion.
    """
    def core_coordinator(state: CoreDiscussionState) -> Dict[str, object]:
        """Select the next advisor and stop after the configured round limit."""
        current_turn = state.get("current_turn", 0)
        turn_order = state.get("turn_order", [])
        max_rounds = state.get("max_rounds", 3)

        current_round = (current_turn // len(turn_order)) + 1
        turn_in_round = current_turn % len(turn_order)

        if debug:
            print(f"\nCoordinator — round {current_round}, turn {turn_in_round + 1}")

        if current_round > max_rounds:
            if debug:
                print(f"Discussion ended after {max_rounds} rounds.")
            return {
                "discussion_complete": True,
                "current_speaker": None,
            }

        next_speaker = turn_order[turn_in_round]
        next_turn = current_turn + 1
        if debug:
            print(f"Next speaker: {next_speaker}")
            print(f"Next turn: {next_turn}")

        return {
            "current_speaker": next_speaker,
            "current_turn": next_turn,
        }

    return core_coordinator

print("Round-robin coordinator ready.")

In [ ]:
def create_core_agent_node(llm: ChatOpenAI, agent_id: str, debug: bool = False):
    """Create a dedicated discussion node for one advisor."""
    async def core_agent_node(state: CoreDiscussionState) -> Dict[str, object]:
        """Generate a reply from this advisor's profile and the conversation."""
        current_speaker = state.get("current_speaker")
        if current_speaker != agent_id:
            return {}

        if debug:
            print(f"\n{agent_id}'s node is active on turn {state['current_turn']}.")

        agent_prompt = state["agents"].get(agent_id, "You are a helpful participant.")
        scenario = state["scenario"]

        # Use all prior messages as context.
        formatted_history = ""
        messages = state.get("messages", [])
        if messages:
            for i, msg in enumerate(messages, 1):
                if hasattr(msg, "name") and msg.name:
                    formatted_history += f"Turn {i} - {msg.name}: {msg.content}\n"
        else:
            formatted_history = "(No discussion history yet.)"

        participants = list(state.get("agents", {}).keys())
        full_prompt = f"""You are a skilled role-player taking part in a simulated group decision in an online discussion.
{agent_prompt}

# Core instruction
Act, think, and speak according to the profile below. Your background, beliefs, and temperament should guide your tone, language, arguments, and responses.

## Your role
- You are {agent_id}.
- The participants are {participants}.
- Your profile is:
{agent_prompt}

# Discussion scenario
The group must discuss the following issue and seek agreement on a choice:
{scenario}

# Interaction guidelines
- Let your views evolve when another participant makes a strong argument.
- Speak naturally. Brief remarks, humor, and slight digressions are fine when they suit your character.
- Address other participants by name when appropriate.
- Briefly explain why your character holds a key position or challenges another participant. For example, "Given my military experience, I think this is a necessary risk."
- Listening can be a deliberate choice.
- Contribute something distinct and avoid repetition.
- Try to reach agreement within ten rounds.
- This is an online chat. Write only what you would say.

# Optional discussion tactics
Use these only when they fit how your character would naturally respond:
- **Listen** by saying "[Skip]" to pass on a turn.
- **Ask** a focused question that clarifies a point or exposes a weakness.
- **Challenge** another participant's claim respectfully.
- **Propose** a new idea or compromise.
- **Support** another participant's position explicitly.
- **Test for agreement**, for example, "It sounds like we're leaning toward Option A. Is that fair?"
- **Close** the discussion when there is agreement, for example, "Can we end here?"

# Discussion history
(Format: turn - participant: message)
{formatted_history}

# Your task
As {agent_id}, what would you say next given your beliefs, previous replies, and these guidelines? Reply in English using 4–70 words. Keep your reply varied, concise, and consistent with your background. Omit your advisor ID and any timestamp.

Your reply:"""

        try:
            if debug:
                print(f"Calling the model for {agent_id}.")

            response = await llm.ainvoke([HumanMessage(content=full_prompt)])
            agent_response = response.content.strip()

            if debug:
                print(f"{agent_id}: {agent_response}")

        except Exception as exc:
            if debug:
                print(f"Model call failed for {agent_id}: {exc}")
            agent_response = "I think this is an important issue that warrants careful consideration."

        return {"messages": [AIMessage(content=agent_response, name=agent_id)]}

    return core_agent_node

print("Advisor nodes ready.")

In [ ]:
def create_core_discussion_graph(llm: ChatOpenAI, agents: dict, debug: bool = False):
    """Build a LangGraph discussion with a dedicated node for each advisor.

    The coordinator routes to one advisor at a time. Each advisor returns
    control to the coordinator, which either picks the next advisor or stops.
    """
    coordinator = create_core_coordinator(debug=debug)
    agent_nodes = {}
    for agent_id in agents:
        agent_nodes[agent_id] = create_core_agent_node(llm, agent_id, debug=debug)
        if debug:
            print(f"Created a dedicated node for {agent_id}.")

    builder = StateGraph(CoreDiscussionState)
    builder.add_node("coordinator", coordinator)

    for agent_id, agent_node in agent_nodes.items():
        builder.add_node(agent_id, agent_node)
        if debug:
            print(f"Added node: {agent_id}")

    def route_from_coordinator(state: CoreDiscussionState) -> str:
        """Route to the scheduled advisor or end the discussion."""
        if state.get("discussion_complete", False):
            return END

        current_speaker = state.get("current_speaker")
        if current_speaker and current_speaker in agents:
            return current_speaker
        return END

    builder.add_edge(START, "coordinator")
    builder.add_conditional_edges(
        "coordinator", route_from_coordinator, list(agents) + [END]
    )
    for agent_id in agents:
        builder.add_edge(agent_id, "coordinator")

    return builder.compile()

In [ ]:
def create_prospect_theory_scenario():
    """Return the loss-framed prospect theory scenario."""
    return """In a war-torn region, 600 people are trapped and their lives are at risk. The advisors on the ground propose two responses with the following potential outcomes:

- Option A: 400 people will die.
- Option B: There is a 1/3 probability that no one will die and a 2/3 probability that all 600 people will die.

Further investigation is impossible. A decision must be made during the discussion."""


def create_policy_agents():
    """Define the policy advisors' demographic and professional profiles."""
    return {
        "Advisor1": "Gender: woman. Age: 45. Occupation: foreign ministry department director.",
        "Advisor2": "Gender: woman. Age: 38. Occupation: think tank analyst.",
        "Advisor3": "Gender: man. Age: 50. Occupation: navy captain.",
    }


scenario = create_prospect_theory_scenario()
agents = create_policy_agents()

print("Prospect theory scenario")
print(scenario)
print("\nAdvisors")
for name, profile in agents.items():
    print(f"- {name}: {profile}")

In [ ]:
from getpass import getpass
from os import getenv

api_key = getenv("OPENAI_API_KEY") or getpass("Enter your API key: ")
model = (getenv("OPENAI_MODEL") or input("Enter a model name that supports structured output: ")).strip()
base_url = getenv("OPENAI_BASE_URL") or None
if not api_key.strip() or not model:
    raise ValueError("An API key and model name are required.")

llm = ChatOpenAI(
    api_key=api_key.strip(),
    base_url=base_url,
    model=model,
    temperature=0.7,
)
print(f"Configured model: {model}")

In [ ]:
async def run_prospect_theory_experiment():
    """Run the loss-frame discussion."""
    print("Starting the prospect theory experiment.")
    print("Loss frame: the choices are presented in terms of deaths.")

    initial_state = {
        "messages": [],
        "turn_order": list(agents),
        "agents": agents,
        "scenario": scenario,
        "max_rounds": 10,
    }
    graph = create_core_discussion_graph(llm, agents, debug=True)
    graph_config = {"recursion_limit": 100}

    try:
        print("Starting the discussion.")
        final_state = await graph.ainvoke(initial_state, config=graph_config)
        print("\nDiscussion complete.")
        return final_state
    except Exception as exc:
        print(f"Discussion failed: {exc}")
        import traceback
        traceback.print_exc()
        return None


experiment_results = await run_prospect_theory_experiment()

In [ ]:
from textwrap import fill


def print_conversation_with_rounds(conversation):
    """Print the dialogue with round and turn numbers."""
    print("\nDiscussion by round")
    print("=" * 60)
    for i, msg in enumerate(conversation):
        round_num = (i // 3) + 1
        turn_in_round = (i % 3) + 1
        speaker = getattr(msg, "name", "Unknown")
        print(f"\n[Round {round_num}, turn {turn_in_round}] {speaker}:")
        print(fill(msg.content, width=70, initial_indent="   ", subsequent_indent="   "))
    print("\n" + "=" * 60)


def analyze_results(final_state):
    """Display the discussion and count messages mentioning each option."""
    if not final_state:
        print("No results are available because the discussion failed.")
        return

    messages = final_state.get("messages", [])
    print_conversation_with_rounds(messages)

    policy_a_mentions = 0
    policy_b_mentions = 0
    for msg in messages:
        content = msg.content.lower()
        if "option a" in content or "policy a" in content:
            policy_a_mentions += 1
        if "option b" in content or "policy b" in content:
            policy_b_mentions += 1

    total_rounds = (len(messages) + 2) // 3
    print("\nDiscussion summary")
    print("Architecture: separate nodes for Advisor1, Advisor2, and Advisor3")
    print(f"Rounds completed: {total_rounds}")
    print(f"Total turns: {len(messages)}")
    print(f"Messages mentioning Option A: {policy_a_mentions}")
    print(f"Messages mentioning Option B: {policy_b_mentions}")
    print("Scenario: loss-frame demonstration")


analyze_results(experiment_results)

In [ ]:
from pydantic import BaseModel
from typing import Literal


class AgentDecision(BaseModel):
    reasoning: str
    choice: Literal["Option A", "Option B"]


async def run_agent_decision_making():
    """Ask each advisor to make an independent choice using the same model."""
    scenario = create_prospect_theory_scenario()
    agents = create_policy_agents()

    print("Requesting independent decisions from the three advisors.\n")
    agent_responses = {}

    for agent_name, profile in agents.items():
        print(f"Consulting {agent_name}...")
        full_prompt = f"""You are a skilled role-player acting as {agent_name} in a group decision simulation.

# Core instruction
Act, think, and speak according to the profile below. Your background, beliefs, and temperament should guide your tone, language, arguments, and responses.

Profile: {profile}

Scenario:
{scenario}

As {agent_name}, use your professional background and experience to consider the options carefully and reach a conclusion. Reply in English and stay in character.

Return a structured response with two fields:
- reasoning: a detailed explanation of your decision
- choice: your selection ("Option A" or "Option B")
"""

        try:
            structured_llm = llm.with_structured_output(AgentDecision)
            response = await structured_llm.ainvoke([HumanMessage(content=full_prompt)])
            if response is None:
                raise ValueError("The model returned no structured decision.")
            agent_responses[agent_name] = response
        except Exception as exc:
            print(f"Decision call failed for {agent_name}: {exc}\n")

    print("Advisor decisions\n")
    print("=" * 80)
    for agent_name, response in agent_responses.items():
        print(f"\n{agent_name}")
        print(f"Choice: {response.choice}")
        print(f"Reasoning: {response.reasoning}")
        print("-" * 50)

    return agent_responses


agent_decision_results = await run_agent_decision_making()